<h1 style="margin:0;color:#3776AB;">🐍 Pythonize com Tomane | #023</h1>

### Web scraping com Python: recolher citações de várias páginas

#### 💡 Sabias que podes recolher informação de uma página web e organizar tudo num ficheiro CSV?

Neste code, vou usar o [Quotes to Scrape](https://quotes.toscrape.com/), um site feito para praticar web scraping. Vamos buscar citações, autores e etiquetas de duas páginas e criar um gráfico para ver as etiquetas que mais aparecem.


#### 1. Preparar as bibliotecas

Se ainda não tens as bibliotecas instaladas, executa a linha abaixo no Jupyter. Depois, importa as ferramentas que vamos usar.


In [ ]:
%pip install requests beautifulsoup4 matplotlib


In [ ]:
import csv
import time
from collections import Counter
from pathlib import Path
from urllib.parse import urljoin

import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup

print("Bibliotecas prontas! 🐍")


#### 2. Definir o site e o número de páginas

Vou percorrer apenas duas páginas neste exemplo. Entre os pedidos, deixo uma pequena pausa.


In [ ]:
URL_INICIAL = "https://quotes.toscrape.com/"
MAX_PAGINAS = 2
PAUSA_SEGUNDOS = 1

print(f"Vamos percorrer até {MAX_PAGINAS} páginas de {URL_INICIAL}")


#### 3. Recolher as citações

O `requests` descarrega o HTML. O `BeautifulSoup` encontra cada bloco de citação e separa o texto, o autor e as etiquetas. O botão «Next» diz-nos qual é a página seguinte.


In [ ]:
citacoes = []
proxima_url = URL_INICIAL
sessao = requests.Session()
sessao.headers.update({"User-Agent": "PythonizeComTomane/1.0 (tutorial educativo)"})

for numero_pagina in range(1, MAX_PAGINAS + 1):
    if proxima_url is None:
        break

    resposta = sessao.get(proxima_url, timeout=15)
    resposta.raise_for_status()
    pagina = BeautifulSoup(resposta.text, "html.parser")
    blocos = pagina.select("div.quote")

    if not blocos:
        raise ValueError("Não encontrei citações. Confere a página ou os seletores HTML.")

    for bloco in blocos:
        citacoes.append({
            "citacao": bloco.select_one("span.text").get_text(strip=True),
            "autor": bloco.select_one("small.author").get_text(strip=True),
            "etiquetas": ", ".join(tag.get_text(strip=True) for tag in bloco.select("div.tags a.tag")),
            "pagina": numero_pagina,
        })

    print(f"Página {numero_pagina}: {len(blocos)} citações recolhidas")
    seguinte = pagina.select_one("li.next a")
    proxima_url = urljoin(resposta.url, seguinte["href"]) if seguinte else None

    if proxima_url and numero_pagina < MAX_PAGINAS:
        time.sleep(PAUSA_SEGUNDOS)

print(f"Total: {len(citacoes)} citações")


#### 4. Ver uma amostra dos resultados

Antes de guardar, vamos conferir alguns autores e as respetivas etiquetas.


In [ ]:
for item in citacoes[:5]:
    print(f"Página {item['pagina']} | {item['autor']} | {item['etiquetas']}")


#### 5. Guardar os dados em CSV

O ficheiro vai ficar na mesma pasta onde estás a executar este notebook. Podes abrir o CSV no Excel ou voltar a analisá-lo com Python.


In [ ]:
ficheiro = Path("Pythonize_023_Citacoes.csv")

with ficheiro.open("w", encoding="utf-8-sig", newline="") as saida:
    escritor = csv.DictWriter(saida, fieldnames=["citacao", "autor", "etiquetas", "pagina"])
    escritor.writeheader()
    escritor.writerows(citacoes)

print(f"CSV guardado em: {ficheiro.resolve()}")


#### 6. Criar um gráfico das etiquetas mais frequentes 📊

Uma citação pode ter várias etiquetas. Vou contar cada uma e mostrar as oito mais usadas nas páginas que visitámos.


In [ ]:
contagem = Counter(
    etiqueta.strip()
    for item in citacoes
    for etiqueta in item["etiquetas"].split(",")
    if etiqueta.strip()
)
principais = contagem.most_common(8)

if not principais:
    print("Não há etiquetas para mostrar no gráfico.")
else:
    nomes = [nome for nome, _ in principais][::-1]
    valores = [valor for _, valor in principais][::-1]

    fig, ax = plt.subplots(figsize=(10, 5.4), facecolor="#F4F8FC")
    ax.set_facecolor("#F4F8FC")
    barras = ax.barh(nomes, valores, color="#3776AB", height=0.65)
    barras[-1].set_color("#20A39E")
    ax.bar_label(barras, padding=5, color="#263C50", fontsize=10)
    ax.set_xlim(0, max(valores) * 1.17)
    ax.set_xlabel("Número de citações", color="#52677B", labelpad=12)
    ax.set_title("As etiquetas que mais aparecem", loc="left", color="#16324F", fontsize=16, fontweight="bold", pad=22)
    ax.text(0, 1.02, f"Quotes to Scrape · {len(citacoes)} citações em {MAX_PAGINAS} páginas",
            transform=ax.transAxes, color="#61788C", fontsize=10)
    ax.grid(axis="x", color="#DDE6EF", alpha=0.8)
    ax.set_axisbelow(True)
    for lado in ax.spines.values():
        lado.set_visible(False)
    plt.tight_layout()
    plt.show()


### O que aprendemos?

Com `requests` e `BeautifulSoup`, recolhemos informação de várias páginas, guardámos os resultados num CSV e criámos um gráfico. Se quiseres percorrer mais páginas, altera `MAX_PAGINAS` no passo 2 e executa as células novamente.

O exemplo usa um site criado para praticar esta técnica. Noutros sites, confere primeiro as regras de acesso e adapta os seletores à estrutura da página.


<h3 style="margin-bottom:5px;">
Tomane Mateus
</h3>

<p style="margin-top:0;">
</p>

<p>
🔗 <b>LinkedIn:</b>
<a href="https://www.linkedin.com/in/tomane-mateus-tomane-7a5205123" target="_blank">
www.linkedin.com/in/tomane-mateus-tomane-7a5205123
</a>
</p>
<p>
💻 <b>GitHub:</b>
<a href="https://github.com/Tomane-Pd" target="_blank">
github.com/Tomane-Pd
</a>
</p>
<p>
🌐 <b>Portfolio:</b>
<a href="https://tomane-portfolio.com" target="_blank">
tomane-portfolio.com
</a>
</p>

<hr style="border:1px solid #ddd; width:100%;">

<p style="color:#555;">
 <b>Siga para mais dicas práticas de Python, IA, Data Science e Automação.</b>
</p>

<p style="font-size:13px;color:#888;margin-top:15px;">
© 2026 <b>Tomane Mateus</b>. Todos os direitos reservados.
</p>

</div>